✓ Imports successful
✓ Database URL loaded: postgresql://postgre...


✓ Success! Design temperature for postal code 81248: -11.30194191°C
  Type: <class 'float'>


✓ Success! Design temperature for postcode 60315: -10.40092384°C


✓ Postal code 81248: -11.30194191°C
✓ Postal code 60315: -10.40092384°C
✓ Postal code 24988: -9.05508179°C
✓ Postal code 93185: -12.97038504°C
✓ Postal code 93489: -13.69747881°C

Summary: 5/5 successful


In [2]:
from heat_load_utils import get_air_change_rate

✓ Year 1977: 0.5 (correct)
✓ Year 1980: 0.5 (correct)
✓ Year 1985: 0.5 (correct)
✓ Year 1990: 0.5 (correct)
✓ Year 1994: 0.5 (correct)


✓ Year 1976: 1.0 (correct)
✓ Year 1970: 1.0 (correct)
✓ Year 1960: 1.0 (correct)
✓ Year 1950: 1.0 (correct)
✓ Year 1900: 1.0 (correct)


Testing boundary values:
✓ Before 1977 - Year 1976: 1.0 (correct)
✓ Start of 1977-1994 range - Year 1977: 0.5 (correct)
✓ End of 1977-1994 range - Year 1994: 0.5 (correct)
✓ Start of 1995+ range - Year 1995: 0.25 (correct)


✓ Expected ValueError caught: Missing required parameter 'year' in JSON data for air change rate calculation




# Testing get_n_walls_touching

## Test 13: Get number of walls touching

In [4]:
# Simple test for get_n_walls_touching
from heat_load_utils import get_n_walls_touching

# Test with valid n_walls_touching value
json_data = {"n_walls_touching": 2}
result = get_n_walls_touching(json_data)
print(f"✓ With n_walls_touching=2: {result}")

# Test with missing key (should return 0)
json_data_missing = {"postal_code": 81248, "year": 1990}
result = get_n_walls_touching(json_data_missing)
print(f"✓ Missing key: {result} (defaults to 0)")

print("\n✓ Test passed!")

✓ With n_walls_touching=2: 2
✓ Missing key: 0 (defaults to 0)

✓ Test passed!


Database connection: ✓ Success

Table: u_values
Columns:
  - id (integer)
  - Code_Construction (text)
  - Code_StatusDataset (text)
  - Code_Country (text)
  - Code_ElementType (text)
  - Code_DataType_Construction (text)
  - Code_Construction_ConstructionYearClass (text)
  - Type_Construction (text)
  - Year1_Construction (text)
  - Year2_Construction (text)
  - U (text)
  - Insulation (text)
  - created_at (timestamp with time zone)
  - updated_at (timestamp with time zone)

✓ Test query successful
Sample row keys: ['Code_ElementType', 'Code_DataType_Construction', 'Code_Country', 'Year1_Construction', 'Year2_Construction', 'U', 'Insulation']
Sample U value: 5.8 (type: str)


In [9]:
# IMPORTANT: Reload module to ensure latest code is used
# Run this cell if you see errors about unquoted column names, year=None issues, or wrong table name
import importlib
import heat_load_utils
importlib.reload(heat_load_utils)
print("✓ Module reloaded")

# Verify the function uses correct table name
import inspect
source = inspect.getsource(heat_load_utils._get_u_value_from_db)
if 'u_value_table_with_insulation' in source:
    print("✓ Function uses correct table: u_value_table_with_insulation")
else:
    print("⚠ Function may be using wrong table name - check source code")
    
# Verify the function signature accepts Optional[int] for year
sig = inspect.signature(heat_load_utils._get_u_value_from_db)
year_param = sig.parameters.get('year')
if year_param and ('Optional' in str(year_param.annotation) or 'None' in str(year_param.annotation)):
    print("✓ Function accepts Optional[int] for year parameter")
else:
    print(f"⚠ Year parameter type: {year_param.annotation if year_param else 'Not found'}")
    
# Check if the function code has year=None handling
if 'if year is not None:' in source:
    print("✓ Function has year=None handling")
else:
    print("⚠ Function may not have year=None handling - check source code")

✓ Module reloaded
✓ Function uses correct table: u_value_table_with_insulation
✓ Function accepts Optional[int] for year parameter
✓ Function has year=None handling


In [11]:
# Test _get_u_value_from_db function directly
# IMPORTANT: Reload module first to ensure latest code with u_value_table_with_insulation
import importlib
import heat_load_utils
importlib.reload(heat_load_utils)

# Import from the correct module path
from heat_load_utils import _get_u_value_from_db
import os
from dotenv import load_dotenv

load_dotenv()
database_url = os.getenv("DATABASE_URL")

print("✓ Module reloaded - using u_value_table_with_insulation")

if database_url:
    print("Test: _get_u_value_from_db function")
    print("=" * 50)
    
    # Test 1: Window - SyAv (note: database uses "SyAv", not "SysAvg")
    print("\n1. Testing Window - SyAv (non-renovated)")
    window_u = _get_u_value_from_db(database_url, 'Window', 'SyAv', 1995)
    print(f"   Year: 1995, Element: Window, Type: SyAv")
    print(f"   ✓ U-value: {window_u} W/(m²·K)" if window_u else "   ✗ No U-value found")
    
    # Test 2: Window - ReEx
    print("\n2. Testing Window - ReEx (renovated)")
    window_u_rex = _get_u_value_from_db(database_url, 'Window', 'ReEx', 1995)
    print(f"   Year: 1995, Element: Window, Type: ReEx")
    print(f"   ✓ U-value: {window_u_rex} W/(m²·K)" if window_u_rex else "   ✗ No U-value found")
    
    # Test 3: Wall - SyAv
    print("\n3. Testing Wall - SyAv (non-renovated)")
    wall_u = _get_u_value_from_db(database_url, 'Wall', 'SyAv', 1995)
    print(f"   Year: 1995, Element: Wall, Type: SyAv")
    print(f"   ✓ U-value: {wall_u} W/(m²·K)" if wall_u else "   ✗ No U-value found")
    
    # Test 4: Wall - ReEx with insulation='yes' (year=None to ignore year filter)
    print("\n4. Testing Wall - ReEx with insulation='yes' (year ignored)")
    wall_u_insulated = _get_u_value_from_db(database_url, 'Wall', 'ReEx', None, insulation='yes')
    print(f"   Year: None (ignored), Element: Wall, Type: ReEx, Insulation: yes")
    print(f"   ✓ U-value: {wall_u_insulated} W/(m²·K)" if wall_u_insulated else "   ✗ No U-value found")
    
    # Test 5: Wall - ReEx without insulation
    print("\n5. Testing Wall - ReEx with insulation='no'")
    wall_u_not_insulated = _get_u_value_from_db(database_url, 'Wall', 'ReEx', 1995, insulation='no')
    print(f"   Year: 1995, Element: Wall, Type: ReEx, Insulation: no")
    print(f"   ✓ U-value: {wall_u_not_insulated} W/(m²·K)" if wall_u_not_insulated else "   ✗ No U-value found")
    
    # Test 6: Roof - SyAv
    print("\n6. Testing Roof - SyAv (non-renovated)")
    roof_u = _get_u_value_from_db(database_url, 'Roof', 'SyAv', 1995)
    print(f"   Year: 1995, Element: Roof, Type: SyAv")
    print(f"   ✓ U-value: {roof_u} W/(m²·K)" if roof_u else "   ✗ No U-value found")
    
    # Test 7: Roof - ReEx with insulation='yes' (year=None to ignore year filter)
    print("\n7. Testing Roof - ReEx with insulation='yes' (year ignored)")
    roof_u_insulated = _get_u_value_from_db(database_url, 'Roof', 'ReEx', None, insulation='yes')
    print(f"   Year: None (ignored), Element: Roof, Type: ReEx, Insulation: yes")
    print(f"   ✓ U-value: {roof_u_insulated} W/(m²·K)" if roof_u_insulated else "   ✗ No U-value found")
    
    # Test 8: Floor - SyAv
    print("\n8. Testing Floor - SyAv (non-renovated)")
    floor_u = _get_u_value_from_db(database_url, 'Floor', 'SyAv', 1995)
    print(f"   Year: 1995, Element: Floor, Type: SyAv")
    print(f"   ✓ U-value: {floor_u} W/(m²·K)" if floor_u else "   ✗ No U-value found")
    
    # Test 9: Floor - ReEx
    print("\n9. Testing Floor - ReEx (renovated)")
    floor_u_rex = _get_u_value_from_db(database_url, 'Floor', 'ReEx', 1995)
    print(f"   Year: 1995, Element: Floor, Type: ReEx")
    print(f"   ✓ U-value: {floor_u_rex} W/(m²·K)" if floor_u_rex else "   ✗ No U-value found")
    
    # Test 10: Different year ranges
    print("\n10. Testing different year ranges")
    years_to_test = [1960, 1975, 1985, 1995, 2005, 2015, 2016]
    for year in years_to_test:
        u_val = _get_u_value_from_db(database_url, 'Window', 'SyAv', year)
        print(f"   Year {year}: {u_val} W/(m²·K)" if u_val else f"   Year {year}: No value found")
    
    # Test 11: Verify return type
    print("\n11. Testing return type")
    test_u = _get_u_value_from_db(database_url, 'Window', 'SyAv', 1995)
    if test_u is not None:
        assert isinstance(test_u, (int, float)), f"Expected numeric type, got {type(test_u)}"
        assert test_u > 0, f"Expected positive value, got {test_u}"
        print(f"   ✓ Return type: {type(test_u).__name__}")
        print(f"   ✓ Value is positive: {test_u}")
    else:
        print("   ⚠ No value returned (might be expected if no data)")
    
    print("\n" + "=" * 50)
    print("✓ All _get_u_value_from_db tests completed!")
    
else:
    print("⚠ DATABASE_URL not found. Skipping database tests.")

✓ Module reloaded - using u_value_table_with_insulation
Test: _get_u_value_from_db function

1. Testing Window - SyAv (non-renovated)
   Year: 1995, Element: Window, Type: SyAv
   ✓ U-value: 1.6 W/(m²·K)

2. Testing Window - ReEx (renovated)
   Year: 1995, Element: Window, Type: ReEx
   ✓ U-value: 3.2 W/(m²·K)

3. Testing Wall - SyAv (non-renovated)
   Year: 1995, Element: Wall, Type: SyAv
   ✓ U-value: 0.39 W/(m²·K)

4. Testing Wall - ReEx with insulation='yes' (year ignored)
   Year: None (ignored), Element: Wall, Type: ReEx, Insulation: yes
   ✓ U-value: 0.35 W/(m²·K)

5. Testing Wall - ReEx with insulation='no'
   Year: 1995, Element: Wall, Type: ReEx, Insulation: no
   ✓ U-value: 1.1 W/(m²·K)

6. Testing Roof - SyAv (non-renovated)
   Year: 1995, Element: Roof, Type: SyAv
   ✓ U-value: 0.34 W/(m²·K)

7. Testing Roof - ReEx with insulation='yes' (year ignored)
   Year: None (ignored), Element: Roof, Type: ReEx, Insulation: yes
   ✓ U-value: 0.35 W/(m²·K)

8. Testing Floor - SyAv (n

Found 1 Wall ReEx entries with Insulation='yes':

1. U-value: 0.35 W/(m²·K)
   Year range: 2010 - 2015
   Type: masonry with 12 cm render insulation system


True

Test 2: With JSON u_values (should use provided values)
✓ U_floor: 0.5 (expected: 0.5)
✓ U_wall: 0.6 (expected: 0.6)
✓ U_roof: 0.35 (expected: 0.35)
✓ U_window: 1.5 (expected: 1.5)

✓ Test 2 passed!


Error: Database error while fetching U-value: column "u" does not exist
LINE 2:         SELECT U
                       ^


✓ Year 1970: 1.0 (type: float)
✓ Year 1985: 0.5 (type: float)
✓ Year 2000: 0.25 (type: float)

✓ All return type validations passed!


---

# Testing get_n_walls_touching

## Test 13: Get number of walls touching

In [16]:
# Test with valid n_walls_touching values
test_cases = [
    {"n_walls_touching": 0},
    {"n_walls_touching": 1},
    {"n_walls_touching": 2},
    {"n_walls_touching": 3},
    {"n_walls_touching": 4},
]

for json_data in test_cases:
    result = get_n_walls_touching(json_data)
    expected = json_data["n_walls_touching"]
    assert result == expected, f"Expected {expected}, got {result}"
    print(f"✓ n_walls_touching={expected}: {result} (correct)")

# Test with missing key (should return 0)
json_data_missing = {"postal_code": 81248, "year": 1990}
result = get_n_walls_touching(json_data_missing)
assert result == 0, f"Expected 0 for missing key, got {result}"
print(f"✓ Missing key: {result} (correct, defaults to 0)")

print("\n✓ All tests passed!")

NameError: name 'get_n_walls_touching' is not defined

✓ Expected ValueError caught: Missing required parameter 'postal_code' in JSON data for design temperature lookup


✓ Expected ValueError caught: No design temperature found in database for postal code: 99999


✓ Return type is float: <class 'float'>
✓ Value -11.30194191°C is within reasonable range (-30°C to 10°C)

✓ All validations passed!
